## Mastering Machine Learning 2025

Taller 2: tokenizers y embeddings

Antes de iniciar abra este cuaderno en Google Colab y habilite la ejecución con GPU:
- En el menú Entorno de ejecución seleccione Cambiar tipo entorno de ejecución.
- Asegúrese de tener seleccionado Python 3.
- Como Acelerador de hardware seleccione GPU T4.

Instale las dependencias para asegurar la correcta ejecución del cuaderno.

In [1]:
!pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

## Punto 1: correr notebook

## Generar texto con prompts y tokens

Ahora usaremos la librería transformers de Hugging Face. Puede crear una cuenta en https://huggingface.co/

Importamos las clases para el tokenizador y el modelo de lenguaje Phi-3-Mini-4K-Instruct, disponible en Hugging Face.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Cargamos el modelo
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)

# Cargamos el tokenizador
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Ejecutemos un prompt solicitando un correo

In [62]:
# Prompt a ejecutar
prompt = "Write an email to authorize the entrance of King William to the University.<|assistant|>"

# Tokenizar el prompt de entrada - retorna tensores de pytorch - los tipos de datos son compatibles con CUDA
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generar el texto
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=40
)

# Imprimir la salida (decodificada)
print(tokenizer.decode(generation_output[0]))

Write an email to authorize the entrance of King William to the University.<|assistant|> Subject: Authorization for King William's Entrance to the University

Dear [Recipient's Name],

I hope this email finds you well. I am


Revisemos los tokens generados como entrada para el modelo, a partir del prompt

In [56]:
input_ids

tensor([[ 4103,  9632,   525,  4013,   770,   338,  9416,   541, 12528, 29915,
           304, 15419, 27139, 29892,  5176, 29892,   322,  5332, 29889, 32001]],
       device='cuda:0')

Revisemos la traducción de tokens a texto (decode)

In [57]:
for id in input_ids[0]:
   print(f'{id}: {tokenizer.decode(id)}')

4103: Trans
9632: late
525: '
4013: This
770: class
338: is
9416: heavy
541: but
12528: cool
29915: '
304: to
15419: Mand
27139: arin
29892: ,
5176: French
29892: ,
322: and
5332: German
29889: .
32001: <|assistant|>


Revisemos la salida en formato token

In [58]:
generation_output

tensor([[ 4103,  9632,   525,  4013,   770,   338,  9416,   541, 12528, 29915,
           304, 15419, 27139, 29892,  5176, 29892,   322,  5332, 29889, 32001,
           448, 15419, 27139, 29901, 29871, 30810, 30502,   235,   178,   193,
         31101,   232,   193,   139, 30908, 30214,   231,   192,   137, 30392,
           232,   193,   139, 30417,   235,   185,   166, 30267, 29898, 29999,
         29882, 15532,   413, 30000, 11555,   865,   298, 30036, 29876,   503]],
       device='cuda:0')

Veamos el tipo de dato de esta salida

In [59]:
generation_output[0][0]

tensor(4103, device='cuda:0')

Veamos ahora la traducción de tokens a texto (decode)

In [60]:
for id in generation_output[0]:
   print(f'{id}: {tokenizer.decode(id)}')

4103: Trans
9632: late
525: '
4013: This
770: class
338: is
9416: heavy
541: but
12528: cool
29915: '
304: to
15419: Mand
27139: arin
29892: ,
5176: French
29892: ,
322: and
5332: German
29889: .
32001: <|assistant|>
448: -
15419: Mand
27139: arin
29901: :
29871: 
30810: 这
30502: 个
235: �
178: �
193: �
31101: 程
232: �
193: �
139: �
30908: 重
30214: ，
231: �
192: �
137: �
30392: 是
232: �
193: �
139: �
30417: 有
235: �
185: �
166: �
30267: 。
29898: (
29999: Z
29882: h
15532: ège
413: k
30000: è
11555: ché
865: ng
298: h
30036: ě
29876: n
503: z


## Comparar tokenizadores

Usaremos la siguiente función para ilustrar el resultado de la tokenización

In [61]:
colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

Usaremos este texto de prueba

In [10]:
text = """

English and CAPITALIZATION

🎵鸟
show_tokens False None elif == >= else: two tabs:" " Three tabs: "   "

12.0*50=600

"""

Inicialmente usaremos BERT en su versión uncased (2018)

In [11]:
show_tokens(text, "google-bert/bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 [SEP] 

Ahora usaremos BERT en su versión cased (2018)

In [12]:
show_tokens(text, "google-bert/bert-base-cased")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 [SEP] 

Ahora usamos GPT-2 (2019)

In [13]:
show_tokens(text, "openai-community/gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


 
 English  and  CAP ITAL IZ ATION 
 
 � � � � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"  "  Three  tabs :  "      " 
 
 12 . 0 * 50 = 600 

 

Ahora usamos Flan-T5 (2022)

In [14]:
show_tokens(text, "google/flan-t5-xxl")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

English and CA PI TAL IZ ATION  <unk> show _ to ken s Fal s e None  e l if = = > = else : two tab s : " " Three tab s : " " 12. 0 * 50 = 600  </s> 

Ahora usamos StarCoder2 (2024)

In [15]:
show_tokens(text, "bigcode/starcoder2-15b")

tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]


 
 English  and  CAPITAL IZATION 
 
 � � � � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"  "  Three  tabs :  "     " 

 1 2 . 0 * 5 0 = 6 0 0 

 

Ahora usamos Phi-3 que usamos al inicio y es similar al de Llama 2

In [16]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


 
 
 English and C AP IT AL IZ ATION 
 
 � � � � � � � 
 show _ to kens False None elif == >= else : two tabs :" " Three tabs : "   " 
 
 1 2 . 0 * 5 0 = 6 0 0 
 
 

## Punto 2

In [72]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

# Función para analizar tokens
def analyze_tokens_and_generate(prompt, model, tokenizer, max_tokens=40):

    print(f"PROMPT: {prompt}")
    print("-" * 50)

    # entrada
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

    print("TOKENS DE ENTRADA:")
    print(f"Tensor shape: {input_ids.shape}")
    print(f"Token IDs: {input_ids[0].tolist()}")
    print("\nDescomposición token por token:")
    for i, token_id in enumerate(input_ids[0]):
        decoded = tokenizer.decode(token_id.item())
        print(f"  {i}: {token_id.item()} -> '{decoded}'")

    # Generar texto
    generation_output = model.generate(
        input_ids=input_ids,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7
    )

    # Analizar salida completa
    print(f"\nTOKENS DE SALIDA:")
    print(f"Tensor shape: {generation_output.shape}")
    full_response = tokenizer.decode(generation_output[0])
    print(f"Respuesta: {full_response}")

    # Tokens nuevos generados
    new_tokens = generation_output[0][len(input_ids[0]):]
    print(f"\nTOKENS NUEVOS GENERADOS ({len(new_tokens)} tokens):")
    for i, token_id in enumerate(new_tokens):
        decoded = tokenizer.decode(token_id.item())
        print(f"  {i}: {token_id.item()} -> '{decoded}'")

    print("\n" + "="*70 + "\n")

    return input_ids, generation_output

# Esta funcion se realiza con ayuda de IA con el fin de presentar resulados organizados y faciles de leer.

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### Prompts

#### Prompt 1

In [73]:
prompt1 = "DESCRIBES THE STEPS TO FOLLOW IN CASE OF HAVING A PANIC ATTACK.<|assistant|>"
print("ANÁLISIS PROMPT 1")
input_ids1, output1 = analyze_tokens_and_generate(prompt1, model, tokenizer, 50)


ANÁLISIS PROMPT 1
PROMPT: DESCRIBES THE STEPS TO FOLLOW IN CASE OF HAVING A PANIC ATTACK.<|assistant|>
--------------------------------------------------
TOKENS DE ENTRADA:
Tensor shape: torch.Size([1, 27])
Token IDs: [23050, 3960, 29933, 2890, 6093, 317, 4330, 7024, 7495, 18322, 2208, 9806, 2672, 29134, 8079, 379, 7520, 4214, 319, 349, 2190, 2965, 15531, 8687, 29968, 29889, 32001]

Descomposición token por token:
  0: 23050 -> 'DESC'
  1: 3960 -> 'RI'
  2: 29933 -> 'B'
  3: 2890 -> 'ES'
  4: 6093 -> 'THE'
  5: 317 -> 'S'
  6: 4330 -> 'TE'
  7: 7024 -> 'PS'
  8: 7495 -> 'TO'
  9: 18322 -> 'FO'
  10: 2208 -> 'LL'
  11: 9806 -> 'OW'
  12: 2672 -> 'IN'
  13: 29134 -> 'CASE'
  14: 8079 -> 'OF'
  15: 379 -> 'H'
  16: 7520 -> 'AV'
  17: 4214 -> 'ING'
  18: 319 -> 'A'
  19: 349 -> 'P'
  20: 2190 -> 'AN'
  21: 2965 -> 'IC'
  22: 15531 -> 'AT'
  23: 8687 -> 'TAC'
  24: 29968 -> 'K'
  25: 29889 -> '.'
  26: 32001 -> '<|assistant|>'

TOKENS DE SALIDA:
Tensor shape: torch.Size([1, 177])
Respuesta:

#### Prompt 2

In [74]:

prompt2 = "Describes the steps to follow in case of having a panic attack.<|assistant|>"

print("ANÁLISIS PROMPT 2 - Texto diferentes idiomas")
input_ids2, output2 = analyze_tokens_and_generate(prompt2, model, tokenizer, 50)


ANÁLISIS PROMPT 2 - Texto diferentes idiomas
PROMPT: Describes the steps to follow in case of having a panic attack.<|assistant|>
--------------------------------------------------
TOKENS DE ENTRADA:
Tensor shape: torch.Size([1, 16])
Token IDs: [20355, 5707, 278, 6576, 304, 1101, 297, 1206, 310, 2534, 263, 7243, 293, 5337, 29889, 32001]

Descomposición token por token:
  0: 20355 -> 'Descri'
  1: 5707 -> 'bes'
  2: 278 -> 'the'
  3: 6576 -> 'steps'
  4: 304 -> 'to'
  5: 1101 -> 'follow'
  6: 297 -> 'in'
  7: 1206 -> 'case'
  8: 310 -> 'of'
  9: 2534 -> 'having'
  10: 263 -> 'a'
  11: 7243 -> 'pan'
  12: 293 -> 'ic'
  13: 5337 -> 'attack'
  14: 29889 -> '.'
  15: 32001 -> '<|assistant|>'

TOKENS DE SALIDA:
Tensor shape: torch.Size([1, 166])
Respuesta: Describes the steps to follow in case of having a panic attack.<|assistant|> 1. Stay calm: Take deep breaths and try to focus on your breathing. Remember that panic attacks do not cause harm and are temporary.

2. Find a quiet space: If po

#### Prompt 3

In [75]:
prompt3 = """Translate 'This class is heavy but cool' to Mandarin, French, and German.<|assistant|>"""

print("ANÁLISIS PROMPT 3 - MATEMÁTICO")
input_ids3, output3 = analyze_tokens_and_generate(prompt3, model, tokenizer, 50)


ANÁLISIS PROMPT 3 - MATEMÁTICO
PROMPT: Translate 'This class is heavy but cool' to Mandarin, French, and German.<|assistant|>
--------------------------------------------------
TOKENS DE ENTRADA:
Tensor shape: torch.Size([1, 20])
Token IDs: [4103, 9632, 525, 4013, 770, 338, 9416, 541, 12528, 29915, 304, 15419, 27139, 29892, 5176, 29892, 322, 5332, 29889, 32001]

Descomposición token por token:
  0: 4103 -> 'Trans'
  1: 9632 -> 'late'
  2: 525 -> '''
  3: 4013 -> 'This'
  4: 770 -> 'class'
  5: 338 -> 'is'
  6: 9416 -> 'heavy'
  7: 541 -> 'but'
  8: 12528 -> 'cool'
  9: 29915 -> '''
  10: 304 -> 'to'
  11: 15419 -> 'Mand'
  12: 27139 -> 'arin'
  13: 29892 -> ','
  14: 5176 -> 'French'
  15: 29892 -> ','
  16: 322 -> 'and'
  17: 5332 -> 'German'
  18: 29889 -> '.'
  19: 32001 -> '<|assistant|>'

TOKENS DE SALIDA:
Tensor shape: torch.Size([1, 170])
Respuesta: Translate 'This class is heavy but cool' to Mandarin, French, and German.<|assistant|> The phrase 'This class is heavy but cool' ca

### Análisis

Es importante resaltar el porque de los prompts elegidos. El 1 y el 2 son iguales pero uno en mayúsculas y el otro en minúsculas respectivamente. Por último, el 3 prompt se realiza para observar como se comporta el modelo en traducciones.

Con respecto a lo anterior se concluye:

Prompts 1 y 2:

La variación de estos dos prompts afecta para que el modelo indique dos respuestas diferentes. Lo anterior se puede asociar a la diferencia de capitalización ya que afecta directamente la tokenizacion y, por ende variaciones entre los tokens de entradas y salidas.

Se puede suponer que en la tokenización de entrada el el prompt 1 algunas palabras al estar en mayúscula se pueden llegar a tratar como nombres propios lo que varia en su asignación de tokens. Lo anterior explicaria la variable en la entrada. Lo anterior se refleja en los tokens de salida y como estos cambian significativamente a pesar de ver solo un cambio que parecería pequeño en la entrada.

Por otro lado, el prompt 3 presenta una traducción por lo que se aleja de la estructura de los dos primeros. En este se observa como las instrucciones de traduccion generan los tokens de entrada, por lo que la respuesta es generada de forma diferente al texto libre o preguntas generales.

## Punto 3

In [23]:
# Función mejorada para mostrar tokens con colores
colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens_detailed(sentence, tokenizer_name):

    print(f"\nTOKENIZADOR: {tokenizer_name}")
    print("-" * 50)

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids

    print(f"Número total de tokens: {len(token_ids)}")
    print("Tokens con colores:")

    # Mostrar con colores
    for idx, t in enumerate(token_ids):
        token_text = tokenizer.decode(t)
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            token_text +
            '\x1b[0m',
            end=' '
        )
    print()  # Nueva línea

    # Mostrar detalle token por token
    print("\nDetalle token por token:")
    for idx, token_id in enumerate(token_ids):
        token_text = tokenizer.decode(token_id)
        print(f"  {idx}: {token_id} -> '{token_text}'")

    return token_ids, len(token_ids)

In [76]:
# Los 3 prompts a comparar
prompts_to_compare = [
    "DESCRIBES THE STEPS TO FOLLOW IN CASE OF HAVING A PANIC ATTACK.<|assistant|>",
    "Describes the steps to follow in case of having a panic attack.<|assistant|>",
    """Translate 'This class is heavy but cool' to Mandarin, French, and German.<|assistant|>"""
]

# Los 3 tokenizadores diferentes a comparar
tokenizers_to_compare = [
    "google-bert/bert-base-cased",      # BERT (2018) - Bidireccional
    "openai-community/gpt2",            # GPT-2 (2019) - Generativo
    "bigcode/starcoder2-15b"            # StarCoder2 (2024) - Para código
]

In [79]:
results_summary = {}

for i, prompt in enumerate(prompts_to_compare, 1):
    print(f"\n{'='*70}")
    print(f"PROMPT {i}: {prompt[:50]}...")
    print(f"{'='*70}")

    results_summary[f"prompt_{i}"] = {}

    # Analizar con cada tokenizador
    for tokenizer_name in tokenizers_to_compare:
        try:
            token_ids, num_tokens = show_tokens_detailed(prompt, tokenizer_name)
            results_summary[f"prompt_{i}"][tokenizer_name] = {
                'num_tokens': num_tokens,
                'token_ids': token_ids
            }
        except Exception as e:
            print(f"Error con {tokenizer_name}: {str(e)}")
            results_summary[f"prompt_{i}"][tokenizer_name] = {
                'num_tokens': 0,
                'error': str(e)
            }
# Esta funcion se realiza con ayuda de IA con el fin de presentar resulados organizados y faciles de leer.


PROMPT 1: DESCRIBES THE STEPS TO FOLLOW IN CASE OF HAVING A ...

TOKENIZADOR: google-bert/bert-base-cased
--------------------------------------------------
Número total de tokens: 37
Tokens con colores:
[CLS] DE ##SC ##RI ##BE ##S THE ST ##EP ##S TO F ##OL ##L ##OW IN CA ##SE OF H ##AV ##ING A PA ##NI ##C AT ##TA ##C ##K . < | assistant | > [SEP] 

Detalle token por token:
  0: 101 -> '[CLS]'
  1: 18581 -> 'DE'
  2: 10844 -> '##SC'
  3: 20595 -> '##RI'
  4: 27211 -> '##BE'
  5: 1708 -> '##S'
  6: 7462 -> 'THE'
  7: 23676 -> 'ST'
  8: 16668 -> '##EP'
  9: 1708 -> '##S'
  10: 16972 -> 'TO'
  11: 143 -> 'F'
  12: 13901 -> '##OL'
  13: 2162 -> '##L'
  14: 17056 -> '##OW'
  15: 15969 -> 'IN'
  16: 8784 -> 'CA'
  17: 12649 -> '##SE'
  18: 11345 -> 'OF'
  19: 145 -> 'H'
  20: 26390 -> '##AV'
  21: 15740 -> '##ING'
  22: 138 -> 'A'
  23: 8544 -> 'PA'
  24: 27451 -> '##NI'
  25: 1658 -> '##C'
  26: 13020 -> 'AT'
  27: 9159 -> '##TA'
  28: 1658 -> '##C'
  29: 2428 -> '##K'
  30: 119 -> '.'
  3

In [78]:
print("\nNÚMERO DE TOKENS POR TOKENIZADOR Y PROMPT:")
print("-" * 50)

for prompt_key in results_summary:
    print(f"\n{prompt_key.upper()}:")
    for tokenizer_name in tokenizers_to_compare:
        if 'error' not in results_summary[prompt_key][tokenizer_name]:
            num_tokens = results_summary[prompt_key][tokenizer_name]['num_tokens']
            print(f"  {tokenizer_name}: {num_tokens} tokens")
        else:
            print(f"  {tokenizer_name}: ERROR")


NÚMERO DE TOKENS POR TOKENIZADOR Y PROMPT:
--------------------------------------------------

PROMPT_1:
  google-bert/bert-base-cased: 37 tokens
  openai-community/gpt2: 28 tokens
  bigcode/starcoder2-15b: 24 tokens

PROMPT_2:
  google-bert/bert-base-cased: 22 tokens
  openai-community/gpt2: 19 tokens
  bigcode/starcoder2-15b: 17 tokens

PROMPT_3:
  google-bert/bert-base-cased: 25 tokens
  openai-community/gpt2: 23 tokens
  bigcode/starcoder2-15b: 23 tokens


Viendo los resultados se observa que para:

Prompt 1 y 2: "Describes the steps to follow in case of having a panic attack" en mayuscula y minuscula

- google-bert/bert-base-cased genera un total de 37 tokens para el Prompt 1 y 22 tokens para el Prompt 2. Observando, BERT utiliza subpalabras para tokenizar, como "##SC", "##RI", agregando en algunas palabras el símbolo ##, que es como para informar que es una continuación de la palabra anterior. Esta tokenización detallada puede resultar en un mayor número de tokens. Los tokens son más largos y detallados, lo que puede aumentar el número de tokens generados, por lo que es un poco más costoso computacionalmente. Por otra parte, manejamos una mayor cantidad de tokens cuando utilizamos el prompt en mayúscula.

- openai-community/gpt2 genera 28 tokens para el Prompt 1 y 19 tokens para el Prompt 2. GPT-2 tiene menos tokens que BERT dado que utiliza como palabras un poco más grandes, como "DES", "C", y "RI", lo que resulta en una tokenización más eficiente y menos fragmentada. La diferencia entre las entradas en mayúsculas y minúsculas no tiene un impacto tan grande como en BERT, pero de igual forma sigue siendo diferente, aun diciendo lo mismo. De igual forma, los signos como , y . se ven que se separan como partes del prompt.

- bigcode/starcoder2-15b genera 24 tokens para el Prompt 1 y 17 tokens para el Prompt 2. La tokenización de Starcoder se ve que utiliza como unidades más grandes, esto hace que hayan menos tokens en total, y los tokens generados se ven como fragmentos más largos, como "DESCR", "IB", y "ES", lo que minimiza la cantidad de tokens. De igual forma, el enfoque de mayúsculas y minúsculas produce diferentes cantidades.


Prompt 3 de traducción de "This class is heavy but cool" a varios idiomas:

- google-bert/bert-base-cased genera 25 tokens, utiliza subpalabras como "Trans" y "##late", lo que hace la tokenización más detallada pero aumenta el número de tokens. Esto es útil para la comprensión, pero genera más fragmentos.

- openai-community/gpt2 genera 23 tokens. Aquí los tokens son como de toda la palabra como "German" y "Mandarin", lo que hace la tokenización más eficiente, pero puede perder algo de precisión en tareas complejas como la traducción.

- bigcode/starcoder2-15b genera tambien 23 tokens. Similar a GPT-2, este modelo utiliza tokens grandes, lo que mejora la eficiencia y es útil para la traducción.

